# Kerangka Agen Microsoft — Azure OpenAI (Responses API)

Dalam contoh kode ini, Anda akan menggunakan **Kerangka Agen Microsoft (MAF)** untuk membuat agen sederhana yang didukung oleh **Azure OpenAI** menggunakan **Responses API**.

> **Catatan migrasi:** Contoh ini sebelumnya menggunakan Semantic Kernel dengan GitHub Models. Kini telah dimigrasikan ke Kerangka Agen Microsoft, dan GitHub Models (yang sudah usang, akan dihentikan Juli 2026) digantikan dengan Azure OpenAI, yang mendukung Responses API. `OpenAIChatClient` di MAF menargetkan endpoint `/openai/v1/` Azure OpenAI yang stabil dan menggunakan Responses API secara default.

Tujuan dari contoh ini adalah untuk mendemonstrasikan langkah-langkah yang nantinya akan diterapkan dalam contoh kode tambahan saat mengimplementasikan berbagai pola agentic.


In [ ]:
%pip install agent-framework agent-framework-openai azure-identity -q


## Impor Paket Python yang Diperlukan


In [ ]:
import os
import random

from dotenv import load_dotenv
from IPython.display import display, HTML

from agent_framework import tool
from agent_framework.openai import OpenAIChatClient
from azure.identity import AzureCliCredential


## Mendefinisikan sebuah Alat

Dalam Microsoft Agent Framework, sebuah **alat** adalah fungsi Python biasa yang dihias dengan `@tool` yang dapat dipanggil oleh agen. Di bawah ini kami mendefinisikan sebuah alat yang mengembalikan tujuan liburan acak dan menghindari mengulang yang sebelumnya.


In [ ]:
# A list of vacation destinations the tool can choose from.
_DESTINATIONS = [
    "Barcelona, Spain",
    "Paris, France",
    "Berlin, Germany",
    "Tokyo, Japan",
    "Sydney, Australia",
    "New York, USA",
    "Cairo, Egypt",
    "Cape Town, South Africa",
    "Rio de Janeiro, Brazil",
    "Bali, Indonesia",
]

# Track the last destination so repeated calls avoid immediate repeats.
_last_destination: str | None = None


@tool(approval_mode="never_require")
def get_random_destination() -> str:
    """Provides a random vacation destination."""
    global _last_destination
    available = _DESTINATIONS.copy()
    if _last_destination and len(available) > 1:
        available.remove(_last_destination)
    destination = random.choice(available)
    _last_destination = destination
    return destination


In [ ]:
load_dotenv()

endpoint = os.environ["AZURE_OPENAI_ENDPOINT"]
deployment = os.environ["AZURE_OPENAI_DEPLOYMENT"]

# OpenAIChatClient targets Azure OpenAI's v1 endpoint and uses the Responses API.
# Sign in with `az login` first so AzureCliCredential can authenticate.
chat_client = OpenAIChatClient(
    model=deployment,
    azure_endpoint=endpoint,
    credential=AzureCliCredential(),
)


## Membuat Agen

Di sini, kita membuat Agen bernama `TravelAgent`.

Dalam contoh ini, kami menggunakan instruksi yang sangat dasar. Silakan modifikasi instruksi-instruksi ini untuk mengamati bagaimana perilaku agen berubah.


In [ ]:
agent = chat_client.as_agent(
    name="TravelAgent",
    instructions="You are a helpful AI Agent that can help plan vacations for customers at random destinations",
    tools=[get_random_destination],
)


## Menjalankan Agen

Sekarang kita dapat menjalankan agen. Kami membuat sebuah `AgentSession` agar agen dapat mengingat percakapan antar giliran, kemudian mengirim dua `user_inputs`. Yang pertama meminta perjalanan; yang kedua mengatakan pengguna tidak menyukai saran tersebut dan meminta yang lain — agen menggunakan riwayat sesi plus alat `get_random_destination` untuk merespons.

Anda dapat memodifikasi pesan-pesan ini untuk mengamati bagaimana agen bereaksi secara berbeda. Respons ditampilkan secara **streaming** token demi token.


In [ ]:
user_inputs = [
    "Plan me a day trip.",
    "I don't like that destination. Plan me another vacation.",
]


async def main():
    # A session keeps conversation history across turns.
    session = agent.create_session()

    for user_input in user_inputs:
        html_output = (
            f"<div style='margin-bottom:10px'>"
            f"<div style='font-weight:bold'>User:</div>"
            f"<div style='margin-left:20px'>{user_input}</div></div>"
        )

        full_response: list[str] = []
        # Stream the agent's response token-by-token. The agent will call the
        # get_random_destination tool automatically when it needs a destination.
        async for chunk in agent.run(user_input, session=session, stream=True):
            full_response.append(str(chunk))

        html_output += (
            "<div style='margin-bottom:20px'>"
            f"<div style='font-weight:bold'>TravelAgent:</div>"
            f"<div style='margin-left:20px; white-space:pre-wrap'>{''.join(full_response)}</div></div><hr>"
        )

        display(HTML(html_output))


await main()


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Penafian**:
Dokumen ini telah diterjemahkan menggunakan layanan terjemahan AI [Co-op Translator](https://github.com/Azure/co-op-translator). Meskipun kami berupaya untuk mencapai akurasi, harap diketahui bahwa terjemahan otomatis mungkin mengandung kesalahan atau ketidakakuratan. Dokumen asli dalam bahasa aslinya harus dianggap sebagai sumber yang sah. Untuk informasi penting, disarankan menggunakan terjemahan profesional oleh manusia. Kami tidak bertanggung jawab atas kesalahpahaman atau penafsiran yang keliru yang timbul dari penggunaan terjemahan ini.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
